# Vespa feed

In [70]:
%env AWS_ENDPOINT_URL=http://minio:9000

env: AWS_ENDPOINT_URL=http://minio:9000


In [76]:
import pandas as pd
from vespa.application import Vespa, ApplicationPackage
from vespa.deployment import VespaDeployment

import os
import pymysql

## Prepare docs

In [77]:
conn = pymysql.connect(
    host=os.getenv('MYSQL_HOST'),
    user=os.getenv('MYSQL_USER'),
    password=os.getenv('MYSQL_PASSWORD'),
    database=os.getenv('MYSQL_DATABASE')
)
print('Connected to MySQL!')

Connected to MySQL!


In [78]:
query = """SELECT
    beers.id as id,
    beers.name as name,
    breweries.name as brewer,
    beers.descript as descript,
    breweries.country as country,
    abv,
    ibu,
    cat_name as category,
    style_name as style
    FROM beers
        LEFT JOIN breweries on beers.brewery_id = breweries.id
        LEFT JOIN categories on beers.cat_id = categories.id
        LEFT JOIN styles on beers.style_id = styles.id
"""
df = pd.read_sql(query, conn)
df.head(3)

/tmp/ipykernel_1692/1119351180.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,id,name,brewer,descript,country,abv,ibu,category,style
0,1,Hocus Pocus,Magic Hat,Our take on a classic summer ale. A toast to ...,United States,4.5,0.0,Other Style,Light American Wheat Ale or Lager
1,2,Grimbergen Blonde,Brouwerij Alken-Maes,,Belgium,6.7,0.0,None,None
2,3,Widdershins Barleywine,Left Hand Brewing Company,,United States,9.1,0.0,None,None


In [79]:
data = df.apply(lambda row: pd.Series({"id": row.id, "fields": row.to_dict()}), axis=1)
data.head(3)

,id,fields
0,1,"{'id': 1, 'name': 'Hocus Pocus', 'brewer': 'Ma..."
1,2,"{'id': 2, 'name': 'Grimbergen Blonde', 'brewer..."
2,3,"{'id': 3, 'name': 'Widdershins Barleywine', 'b..."


## Connect to Vespa

In [72]:
client = Vespa(url="http://vespa", port=19071)
client.wait_for_application_up(120)

Application is up!


### Deploy application package

In [ ]:
from vespa.package import (
    ApplicationPackage,
    Field,
    Schema,
    Document,
    HNSW,
    RankProfile,
    Component,
    Parameter,
    FieldSet,
    GlobalPhaseRanking,
    Function, 
    ServicesConfiguration
)

In [5]:
APP_NAME = "beers"
SCHEMA_NAME = "beer"

In [3]:
schema_beer = Schema(
    name="beer",
    document=Document(
        fields=[
            Field(name="id", type="int", indexing=["summary"]),
            Field(
                name="name",
                type="string",
                indexing=["index", "summary"],
                index="enable-bm25",
            ),
            Field(
                name="brewer",
                type="string",
                indexing=["index", "summary"],
                index="enable-bm25",
                bolding=True,
            ),
            Field(
                name="descript",
                type="string",
                indexing=["index", "summary"],
                index="enable-bm25",
                bolding=True,
            ),
            Field(
                name="country",
                type="string",
                indexing=["summary"],
            ),
            Field(
                name="category",
                type="string",
                indexing=["summary"],
            ),
            Field(
                name="style",
                type="string",
                indexing=["summary"],
            ),
            Field(
                name="abv",
                type="float",
                indexing=["summary"],
            ),
            Field(
                name="ibu",
                type="float",
                indexing=["summary"],
            ),
            Field(
                name="embedding",
                type="tensor(x[384])",
                indexing=[
                    'input name . " " . input descript',
                    "embed",
                    "index",
                    "attribute",
                ],
                ann=HNSW(distance_metric="angular"),
                is_document_field=False,
            ),
        ]
    ),
    fieldsets=[FieldSet(name="default", fields=["name", "descript"])],
    rank_profiles=[
        RankProfile(
            name="bm25",
            inputs=[("query(q)", "tensor(x[384])")],
            functions=[
                Function(name="bm25sum", expression="bm25(name) + bm25(descript)")
            ],
            first_phase="bm25sum",
        ),
        RankProfile(
            name="semantic",
            inputs=[("query(q)", "tensor(x[384])")],
            first_phase="closeness(field, embedding)",
        ),
        RankProfile(
            name="fusion",
            inherits="bm25",
            inputs=[("query(q)", "tensor(x[384])")],
            first_phase="closeness(field, embedding)",
            global_phase=GlobalPhaseRanking(
                expression="reciprocal_rank_fusion(bm25sum, closeness(field, embedding))",
                rerank_count=1000,
            ),
        ),
    ],
)

In [62]:
from vespa.package import ServicesConfiguration
from vespa.configuration.services import (
    services,
    container,
    search,
    document_api,
    document_processing,
    content,
    redundancy,
    documents,
    document,
    node,
    nodes, 
    tuning, 
    resource_limits,
    disk,
    component, transformer_model, tokenizer_model
)

services_config = ServicesConfiguration(
    application_name="beers",
    schemas=[schema],
    services_config=services(
        container(id="beers_container", version="1.0")(
            search(),
            document_api(),
            document_processing(),
            component(id="e5", type="hugging-face-embedder")(
                transformer_model(
                    url="https://data.vespa-cloud.com/sample-apps-data/e5-small-v2-int8/e5-small-v2-int8.onnx"
                ),
                tokenizer_model(
                    url="https://data.vespa-cloud.com/sample-apps-data/e5-small-v2-int8/tokenizer.json"
                ),
            ),
        ),
        content(id="beers_content", version="1.0")(
            redundancy("1"),
            documents(document(type="beer", mode="index")),
            nodes(node(distribution_key="0", hostalias="node1")),
            tuning(resource_limits(disk("0.9"))),
        ),
        version="1.0",
    ),
)

In [63]:
package = ApplicationPackage(
    name="beers",
    schema=[schema_beer],
    components=[component_beer], 
    services_config=services_config
)

In [64]:
package.to_files("./vespa")

In [65]:
package.to_zipfile("./vespa.zip")

In [66]:
!curl --header Content-Type:application/zip \
  --data-binary @vespa.zip \
  http://vespa:19071/application/v2/tenant/default/prepareandactivate

{"log":[],"message":"Session 6 for tenant 'default' prepared and activated.","session-id":"6","activated":true,"tenant":"default","url":"http://vespa:19071/application/v2/tenant/default/application/default/environment/prod/region/default/instance/default","configChangeActions":{"restart":[],"refeed":[],"reindex":[]}}

### Feed

In [ ]:
curl -X PUT -H "Content-Type:application/json" --data '
  {
      "fields": {
          "name": {
              "assign": "toto"
          }
      }
  }' \
  http://vespa/document/v1/beers/beer/docid/1

In [83]:
from loguru import logger
from vespa.io import VespaResponse


class VespaFeederCallback:
    def __init__(self):
        self.total_documents = 0
        self.failed_documents_ids = []

    def record_callback(self, response: VespaResponse, id_: str):
        self.total_documents += 1
        if not response.is_successful():
            self.failed_documents_ids.append(id_)

    def summary(self):
        logger.info(f"Total documents sent: {self.total_documents}")
        logger.info(f"Total documents succeeded: {self.total_documents - len(self.failed_documents_ids)}")
        if len(self.failed_documents_ids) > 0:
            logger.warning(f"Total documents failed: {len(self.failed_documents_ids)}")

callback = VespaFeederCallback()

In [74]:
client = Vespa(url="http://vespa", port=8080)

In [86]:
client.feed_iterable(
    data.to_dict(orient="records"),
    schema="beer",
    callback=callback.record_callback
)

In [87]:
callback.summary()

2026-01-02 10:59:27.585 | INFO     | __main__:summary:16 - Total documents sent: 5917
2026-01-02 10:59:27.588 | INFO     | __main__:summary:17 - Total documents succeeded: 5917
